In [1]:
import numpy as np

class TorqueLimiter:
    def __init__(self, kl_rfmin_krstsys_map):
        """
        kl_rfmin_krstsys_map: Dictionary or interpolation function
        representing the Nkw-dependent baseline curve.
        """
        self.kl_rfmin_krstsys_map = kl_rfmin_krstsys_map
        # Initial state for the unit delay block (1/z)
        self.drf_max_ti_prev = 327.67

    def get_baseline(self, nkw):
        """Simulates the KL_RFMIN_KRSTSYS lookup table."""
        # Simple linear interpolation based on engine speed (Nkw)
        speeds = sorted(self.kl_rfmin_krstsys_map.keys())
        values = [self.kl_rfmin_krstsys_map[s] for s in speeds]
        return np.interp(nkw, speeds, values)

    def calculate_step(self, rf, nkw, tq_lim_active, err_hpp_sfu, hpp_max_uw):
        """
        Executes one calculation step of the logic diagram.
        """
        # 1. Baseline from Lookup Table
        rf_baseline = self.get_baseline(nkw)

        # 2. Determine Ramping Gradients (Switch logic)
        if err_hpp_sfu:
            ramp_neg = 0.5  # Placeholder for K_DRF_RFMAXRAMPEGW_NEG
            ramp_pos = 0.1  # Placeholder for K_DRF_RFMAXRAMPEGW_POS
        else:
            ramp_neg = 0.2  # Placeholder for K_DRF_RFMAXRAMPE_NEG
            ramp_pos = 0.05 # Placeholder for K_DRF_RFMAXRAMPE_POS

        # 3. Delta Logic (The core 1/z feedback loop)
        if tq_lim_active:
            # Limitation requested: Decrease Delta-Rf
            # Compare current Delta (Rf - baseline) with previous ramp state
            current_delta_rf = rf - rf_baseline
            target_delta = min(current_delta_rf, self.drf_max_ti_prev - ramp_neg)
        else:
            # Limitation inactive: Increase Delta-Rf (Recovery)
            target_delta = self.drf_max_ti_prev + ramp_pos

        # Ensure Delta-Rf doesn't go below 0
        self.drf_max_ti_prev = max(0, target_delta)

        # 4. Final Output Summation
        # Calculate primary limit: Baseline + Delta-Rf
        rf_max_ti_calculated = rf_baseline + self.drf_max_ti_prev

        # Secondary safety path (bottom branch of diagram)
        rf_safety_min = max(rf_baseline, rf * hpp_max_uw)

        # Final Min-Selection
        rf_max_ti = min(rf_max_ti_calculated, rf_safety_min)

        return self.drf_max_ti_prev, rf_max_ti

# --- Example Usage ---
# RPM vs Minimum Charge Baseline
map_data = {100: 300, 7000: 300}
limiter = TorqueLimiter(map_data)

# Simulate a step where limitation is active at 2000 RPM
drf, rf_limit = limiter.calculate_step(
    rf=200.0,
    nkw=2000,
    tq_lim_active=True,
    err_hpp_sfu=False,
    hpp_max_uw=1.2
)

print(f"Current Delta (Drf_max_ti): {drf}")
print(f"Final Charge Limit (Rf_max_ti): {rf_limit}")



Current Delta (Drf_max_ti): 0
Final Charge Limit (Rf_max_ti): 300.0


In [2]:
print(f"{'Nkw':<6} | {'Rf In':<6} | {'Rf Max Limit':<12}")
print("-" * 40)

for i in range(1, 11):
    # Simulate a slight increase in engine speed and driver demand (Rf)
    current_nkw = 2500 + (i * 100)
    current_rf_in = 60.0 + (i * 0.5)
    
    # Invoke the limiter
    drf_output, rf_max_ti = limiter.calculate_step(
        rf = current_rf_in, nkw = current_nkw, tq_lim_active = True, err_hpp_sfu=False, hpp_max_uw = 1.2)
    
    print(f"{current_nkw:<6} | {current_rf_in:<6.1f} | {rf_max_ti:<12.2f}")
    

Nkw    | Rf In  | Rf Max Limit
----------------------------------------
2600   | 60.5   | 300.00      
2700   | 61.0   | 300.00      
2800   | 61.5   | 300.00      
2900   | 62.0   | 300.00      
3000   | 62.5   | 300.00      
3100   | 63.0   | 300.00      
3200   | 63.5   | 300.00      
3300   | 64.0   | 300.00      
3400   | 64.5   | 300.00      
3500   | 65.0   | 300.00      
